# PGMPy Datasets API

A guided exploration of pgmpy's dataset loading subsystem:
- `list_datasets()`: Discover available datasets with rich filters
- `load_dataset()`: Load a dataset by name into a structured `Dataset` object
- `Dataset` attributes: tabular data, ground-truth DAG, expert knowledge, metadata tags

This module is the entry point for obtaining benchmark data used in causal
discovery, structure learning, and parameter estimation.

## Imports and Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import logging
import sys
import warnings

warnings.filterwarnings('ignore')

In [ ]:
import pgmpy.datasets as ds
import pandas as pd
import numpy as np

In [ ]:
# Use this for most notebooks.
import helpers.htutorial as htutori

htutori.config_notebook()

# Import pgmpy utilities.
import tutorials.pgmpy.pgmpy_utils as tpgpguti

# Initialize logger.
logging.basicConfig(level=logging.INFO)
_LOG = logging.getLogger(__name__)

# Convert `display` into `print()` when running outside IPython.
try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore

# Part 1: Library Overview

## What problem does `pgmpy.datasets` solve?

- Provides one-line access to 47+ curated benchmark datasets for graphical models
- Eliminates the boilerplate of finding, downloading, and parsing standard datasets
- Some datasets include the true causal graph (ground truth) for supervised evaluation
- Some include expert domain knowledge (edge constraints) for causal discovery

## Key abstractions

- **Catalog**: `list_datasets()` returns a list of dataset name strings
  - Supports keyword filters: `is_discrete`, `is_continuous`, `has_ground_truth`, etc.
- **Loader**: `load_dataset(name)` returns a single `Dataset` object
- **Dataset**: A lightweight dataclass with five key fields:
  - `name` (str): Dataset identifier
  - `data` (pd.DataFrame): The actual tabular data
  - `ground_truth` (DAG or None): True causal graph (when available)
  - `expert_knowledge` (ExpertKnowledge or None): Domain edge constraints
  - `tags` (dict): Metadata (n_variables, n_samples, data types, etc.)

## How the pieces fit together

```
User -> list_datasets(is_discrete=True) -> ['college_plans', ...]
User -> load_dataset('sachs_discrete')
     -> Dataset(name='sachs_discrete',
                data=DataFrame(5400 x 11),
                ground_truth=DAG(11 nodes, 20 edges),
                tags={n_variables: 11, ...})
```

# Part 2: Primitive-by-Primitive Exploration

## Primitive 1: `list_datasets()` — The Catalog

**Mental model**: A searchable registry of all available benchmark datasets.
Returns a plain list of name strings suitable for passing to `load_dataset()`.

In [ ]:
# Smallest construction: no arguments returns all 47+ datasets.
all_datasets = ds.list_datasets()
print(f"Total datasets available: {len(all_datasets)}")
print(f"First 10: {all_datasets[:10]}")

In [ ]:
# Inspect the object.
print(f"type: {type(all_datasets)}")
print(f"length: {len(all_datasets)}")
print(f"all strings: {all(isinstance(n, str) for n in all_datasets)}")

### `list_datasets()` Filters

The real power comes from keyword filters that narrow the catalog.

In [ ]:
# Filter by data type: discrete only.
discrete_ds = ds.list_datasets(is_discrete=True)
print(f"Discrete datasets ({len(discrete_ds)}): {discrete_ds}")

In [ ]:
# Filter by data type: continuous only.
continuous_ds = ds.list_datasets(is_continuous=True)
print(f"Continuous datasets ({len(continuous_ds)}): {continuous_ds[:10]}")

In [ ]:
# Filter by data type: mixed (both discrete and continuous columns).
mixed_ds = ds.list_datasets(is_mixed=True)
print(f"Mixed datasets ({len(mixed_ds)}): {mixed_ds}")

**Question**: Can a dataset be both discrete and mixed?

In [ ]:
# Check: datasets that match both discrete and mixed filters.
both = ds.list_datasets(is_discrete=True, is_mixed=True)
print(f"Both discrete and mixed: {len(both)} datasets")
# A dataset is either discrete, continuous, or mixed — never more than one.

In [ ]:
# Filter by presence of ground truth.
with_gt = ds.list_datasets(has_ground_truth=True)
print(f"Datasets with ground truth ({len(with_gt)}): {with_gt}")

In [ ]:
# Filter by number of variables.
few_vars = ds.list_datasets(n_variables=5)
print(f"Datasets with exactly 5 variables: {few_vars}")

ten_vars = ds.list_datasets(n_variables=10)
print(f"Datasets with exactly 10 variables: {ten_vars}")

**Question**: What happens with a filter that matches no datasets?

In [ ]:
# Empty result for very restrictive filters.
empty = ds.list_datasets(n_samples=100, is_discrete=True)
print(f"Restrictive filter result: {empty}")
# Returns an empty list — no error.

In [ ]:
# Combine multiple filters for precise selection.
rare = ds.list_datasets(is_interventional=True)
print(f"Interventional datasets ({len(rare)}): {rare}")

simulated = ds.list_datasets(is_simulated=True)
print(f"Simulated datasets ({len(simulated)}): {simulated}")

## Primitive 2: `load_dataset()` — The Loader

**Mental model**: Given a name string from the catalog, load and return a fully
populated `Dataset` object with tabular data and optional metadata.

In [ ]:
# Smallest construction: load a simple dataset.
data = ds.load_dataset('galton_stature')
print(f"Loaded dataset: {data.name}")
print(f"Object type: {type(data)}")

In [ ]:
# Inspect the object.
print(f"type(data): {type(data)}")
print(f"dir(data) basic attributes:", [a for a in dir(data) if not a.startswith('_')])

In [ ]:
# Try loading a different dataset.
data2 = ds.load_dataset('pima_diabetes')
print(f"Loaded: {data2.name}")
print(f"type: {type(data2)}")
print(f"dir: {[a for a in dir(data2) if not a.startswith('_')]}")

## Primitive 3: `Dataset.data` — The Tabular Data

**Mental model**: The actual observations as a pandas DataFrame, ready for
`estimator.fit(data)`.

In [ ]:
# Load the Pima diabetes dataset.
data = ds.load_dataset('pima_diabetes')

# Inspect the data field.
print(f"type: {type(data.data)}")
print(f"shape: {data.data.shape}")
print(f"columns: {list(data.data.columns)}")
print(f"dtypes:\n{data.data.dtypes}")
print(f"\nfirst 5 rows:")
display(data.data.head())

In [ ]:
# Check basic statistics.
print(f"Missing values: {data.data.isnull().sum().sum()}")
print(f"Summary statistics:")
display(data.data.describe())

## Primitive 4: `Dataset.ground_truth` — The True Causal Graph

**Mental model**: When available, this is a `DAG` object representing the true
underlying causal structure. Used to evaluate how well a causal discovery
algorithm recovered the correct graph.

In [ ]:
# Load a dataset with ground truth.
data = ds.load_dataset('sachs_discrete')

# Inspect the ground truth.
print(f"ground_truth type: {type(data.ground_truth)}")
print(f"Number of nodes: {len(data.ground_truth.nodes())}")
print(f"Number of edges: {len(data.ground_truth.edges())}")
print(f"\nNodes: {sorted(list(data.ground_truth.nodes()))}")
print(f"\nEdges: {list(data.ground_truth.edges())}")

In [ ]:
# Inspect the DAG: check predecessors and successors.
gt = data.ground_truth
print(f"Predecessors of 'erk': {list(gt.predecessors('erk'))}")
print(f"Successors of 'pkc': {list(gt.successors('pkc'))}")
print(f"Is 'pkc' a parent of 'jnk'?: {gt.has_edge('pkc', 'jnk')}")
print(f"Is 'jnk' a parent of 'pkc'?: {gt.has_edge('jnk', 'pkc')}")

In [ ]:
# Check what happens when a dataset has no ground truth.
data2 = ds.load_dataset('galton_stature')
print(f"galton_stature ground_truth: {data2.ground_truth}")
# Returns None — the field is optional.

## Primitive 5: `Dataset.expert_knowledge` — Domain Constraints

**Mental model**: Prior knowledge about edges that must be present, must be
absent, or temporal ordering — used to guide causal discovery algorithms.

In [ ]:
# Load a dataset with expert knowledge.
data = ds.load_dataset('sachs_discrete')

# Inspect expert knowledge.
ek = data.expert_knowledge
print(f"expert_knowledge type: {type(ek)}")
print(f"forbidden_edges: {ek.forbidden_edges}")
print(f"required_edges count: {len(ek.required_edges)}")
print(f"required_edges (sample): {list(ek.required_edges)[:5]}")
print(f"search_space: {ek.search_space}")
print(f"temporal_order: {ek.temporal_order}")

In [ ]:
# Check dataset without expert knowledge.
data2 = ds.load_dataset('galton_stature')
print(f"galton_stature expert_knowledge: {data2.expert_knowledge}")

## Primitive 6: `Dataset.tags` — Metadata Dictionary

**Mental model**: A dictionary of descriptive metadata about the dataset,
including counts, boolean flags for data type, and provenance info.

In [ ]:
# Load a dataset and inspect tags.
data = ds.load_dataset('sachs_discrete')
print(f"tags type: {type(data.tags)}")

# Print each tag key-value pair.
for key, value in data.tags.items():
    print(f"  {key}: {value}")

In [ ]:
# Compare tags across different dataset types.
sachs_tags = ds.load_dataset('sachs_discrete').tags
galton_tags = ds.load_dataset('galton_stature').tags

print(f"{'Property':<25} {'sachs_discrete':<20} {'galton_stature':<20}")
print("-" * 65)
for key in sachs_tags:
    print(f"{key:<25} {str(sachs_tags[key]):<20} {str(galton_tags[key]):<20}")

# Part 3: Composition Examples

## Example 1: List a Category, Load the First Dataset, Inspect It

Minimal workflow: discover -> load -> explore.

In [ ]:
# Discover discrete datasets.
discrete = ds.list_datasets(is_discrete=True)
print(f"Discrete datasets: {discrete}")

# Load the first one.
ds1 = ds.load_dataset(discrete[0])
print(f"\nName: {ds1.name}")
print(f"Shape: {ds1.data.shape}")
print(f"Columns: {list(ds1.data.columns)}")
display(ds1.data.head())

# Print tags.
for k, v in ds1.tags.items():
    print(f"  {k}: {v}")

## Example 2: Filter by Ground Truth, Load, Compare Data vs Graph

In [ ]:
gt_ds = ds.list_datasets(has_ground_truth=True)
print(f"Datasets with ground truth: {gt_ds}")

# Use `sachs_discrete` since some ground-truth graphs have parsing issues.
ds2 = ds.load_dataset('sachs_discrete')

print(f"Dataset: {ds2.name}")
print(f"Data shape: {ds2.data.shape}")
print(f"Ground truth nodes: {len(ds2.ground_truth.nodes())}")
print(f"Ground truth edges: {len(ds2.ground_truth.edges())}")

# Show a few rows.
display(ds2.data.head(3))

# Show the graph edges.
print(f"Edges: {list(ds2.ground_truth.edges())}")

## Example 3: Load a Large Continuous Dataset and Summarize

In [ ]:
# Find continuous datasets.
cont = ds.list_datasets(is_continuous=True)
print(f"Continuous datasets: {len(cont)}")
print(f"  {cont}")

# Load a modest-sized one.
ds3 = ds.load_dataset('airfoil')
print(f"\nDataset: {ds3.name}")
print(f"Shape: {ds3.data.shape}")
print(f"Columns: {list(ds3.data.columns)}")
display(ds3.data.describe())

## Example 4: Use Expert Knowledge + Ground Truth Together

The Sachs dataset family includes both ground truth and expert knowledge,
making it useful for benchmarking causal discovery algorithms.

In [ ]:
# Load the mixed-type Sachs dataset.
ds4 = ds.load_dataset('sachs_mixed')

print(f"Dataset: {ds4.name}")
print(f"Data type: discrete, continuous, or mixed?")
print(f"  is_discrete: {ds4.tags['is_discrete']}")
print(f"  is_continuous: {ds4.tags['is_continuous']}")
print(f"  is_mixed: {ds4.tags['is_mixed']}")
print(f"\nShape: {ds4.data.shape}")

# Check ground truth structure.
gt = ds4.ground_truth
print(f"\nGround truth: {gt.nodes()} -> {gt.edges()}")

# Check expert knowledge.
ek = ds4.expert_knowledge
print(f"\nExpert knowledge:")
print(f"  Required edges: {len(ek.required_edges)}")
print(f"  Forbidden edges: {len(ek.forbidden_edges)}")

In [ ]:
# Check that the same ground truth graph is shared across all Sachs variants.
sachs_variants = [n for n in ds.list_datasets() if n.startswith('sachs')]
print(f"Sachs variants: {sachs_variants}")

gt_edges = {}
for name in sachs_variants:
    try:
        d = ds.load_dataset(name)
        gt_edges[name] = len(d.ground_truth.edges())
        print(f"  {name}: {d.data.shape}, GT edges: {len(d.ground_truth.edges())}")
    except Exception as e:
        print(f"  {name}: ERROR - {type(e).__name__}")

# All Sachs variants share the same ground truth graph.
all_same = len(set(gt_edges.values())) == 1
print(f"\nAll variants share the same graph? {all_same}")

# Part 4: API Patterns

## 1. Catalog-Load Pattern

The dominant pattern: `list_datasets()` (discover) -> `load_dataset()` (fetch).
This is similar to `sklearn.datasets.fetch_*` or PyTorch's `torchvision.datasets`.

In [ ]:
# The pattern in one line.
data = ds.load_dataset(ds.list_datasets(is_discrete=True)[0])
print(f"Loaded: {data.name}")

## 2. Filter Criterion Pattern

Filters are keyword arguments that function as predicates on the dataset metadata.
Multiple filters combine with AND semantics.

In [ ]:
# AND-combined filters: discrete AND has ground truth AND expert knowledge.
precise = ds.list_datasets(is_discrete=True, has_ground_truth=True)
print(f"Discrete datasets with ground truth: {precise}")

## 3. Dataset as a Dataclass

The `Dataset` object bundles data + metadata into one container:
- `data`: Always present (pd.DataFrame)
- `ground_truth`: Optional (DAG or None)
- `expert_knowledge`: Optional (ExpertKnowledge or None)
- `tags`: Always present (dict)

In [ ]:
# Show the dataclass structure via a print.
print(ds.load_dataset('boston_housing'))

## 4. Optional Ground Truth / Expert Knowledge

Not all datasets provide ground truth or expert knowledge. Always check for
`None` before using them.

In [ ]:
# Safe access pattern.
# Note: some datasets require HuggingFace download which may fail in restricted
# environments. We catch those gracefully.
def summarize_dataset(name):
    try:
        d = ds.load_dataset(name)
    except Exception as e:
        print(f"{name:30s} | ERROR: {type(e).__name__}")
        return
    has_gt = d.ground_truth is not None
    has_ek = d.expert_knowledge is not None
    print(f"{name:30s} | shape={str(d.data.shape):15s} | GT={has_gt} | EK={has_ek}")

for name in ds.list_datasets()[:10]:
    summarize_dataset(name)

# Part 5: Interactive Exploration

Experiment with the API yourself.

In [ ]:
# Explore all available filters.
print("All list_datasets() keyword arguments:")
import inspect
sig = inspect.signature(ds.list_datasets)
print(f"  {sig}")

In [ ]:
# Question: What happens if you pass an invalid dataset name?
try:
    ds.load_dataset('nonexistent_dataset')
except Exception as e:
    print(f"Error type: {type(e).__name__}")
    print(f"Error message: {e}")

In [ ]:
# Question: What happens if you pass an invalid filter name?
try:
    ds.list_datasets(invalid_filter=True)
except Exception as e:
    print(f"Error type: {type(e).__name__}")
    print(f"Error message: {e}")

In [ ]:
# Explore: list all datasets and their tags in a table.
# Some datasets require HuggingFace download and may fail — skip those.
rows = []
for name in ds.list_datasets():
    try:
        d = ds.load_dataset(name)
    except Exception:
        continue
    rows.append({
        'name': name,
        'n_vars': d.tags['n_variables'],
        'n_samples': d.tags['n_samples'],
        'type': 'disc' if d.tags['is_discrete'] else ('cont' if d.tags['is_continuous'] else 'mixed'),
        'has_GT': d.tags['has_ground_truth'],
        'has_EK': d.tags['has_expert_knowledge'],
        'simulated': d.tags['is_simulated'],
    })

summary = pd.DataFrame(rows)
print(f"Loaded {len(rows)} of {len(ds.list_datasets())} datasets successfully")
display(summary)

In [ ]:
# Question: What is the range of dataset sizes?
if len(summary) > 0:
    print(f"Minimum variables: {summary['n_vars'].min()}")
    print(f"Maximum variables: {summary['n_vars'].max()}")
    print(f"Minimum samples: {summary['n_samples'].min()}")
    print(f"Maximum samples: {summary['n_samples'].max()}")
    print(f"Median samples: {summary['n_samples'].median()}")
else:
    print("No datasets loaded successfully to compute statistics")

In [ ]:
# Question: Which datasets have NO ground truth AND NO expert knowledge?
if len(summary) > 0:
    plain = summary[(summary['has_GT'] == False) & (summary['has_EK'] == False)]
    print(f"Datasets with neither GT nor EK: {len(plain)}")
    display(plain.head())
else:
    print("No summary data available")

## Summary: The Mental Model

`pgmpy.datasets` implements a **catalog-loader** pattern:
- `list_datasets(**filters)` discovers available datasets by querying a metadata
  registry, returning matching name strings
- `load_dataset(name)` fetches the named dataset and returns a `Dataset` dataclass
  that bundles a `pd.DataFrame` (`.data`) with optional resources:
  - A true causal graph (`.ground_truth`) for supervised structure learning evaluation
  - Domain constraints (`.expert_knowledge`) to guide discovery algorithms
  - Descriptive metadata (`.tags`) for programmatic filtering and inspection
- The separation of catalog from loader, combined with keyword filters, makes the
  API scalable: users can discover without loading, and load only what they need.